In [104]:
import pandas as pd
import pytz
import datetime as dt
from sklearn.linear_model import LinearRegression
import joblib
from sklearn.preprocessing import StandardScaler

In [105]:
def preprocess_QuantAQ(file_path, start_date, end_date):
    sensor = pd.read_csv(file_path, parse_dates=['timestamp'])
    sensor = sensor[['timestamp','rh','temp','o3']].dropna(subset=['o3'])
    sensor.rename(columns={'timestamp':'time'}, inplace=True)
    
    sensor['time'] = pd.to_datetime(sensor['time'])
    est = pytz.timezone('US/Eastern')
    sensor['time'] = sensor['time'].dt.tz_convert(est)
    sensor['time'] = sensor['time'].dt.tz_localize(None)
    sensor['day'] = sensor['time'].dt.date
    sensor['dayhour'] = sensor['time'].dt.strftime('%Y-%m-%d %H')
    
    sensor = sensor[(sensor['time'] >= start_date) & (sensor['time'] <= end_date)]


    sensor = sensor[::-1].reset_index(drop=True)
    # train only on last 3/4 of data
    # split_index = int(len(sensor) * 0.25)
    # sensor = sensor.iloc[split_index:]


    return sensor


In [106]:
# returns a model that predicts daily o3 from hourly sensor data
def train_hourly_model(sensor_collocation_data, h_regular):
    temp = sensor_collocation_data.groupby('dayhour').agg(
        shourlyO3=('o3', lambda x: x.mean(skipna=True)),
        shourlyRH=('rh', lambda x: x.mean(skipna=True)),
        shourlyTemp=('temp', lambda x: x.mean(skipna=True))
    ).reset_index()

    # inner join (default) - only keep rows that are in both dataframes
    collocation_data = pd.merge(h_regular, temp, on='dayhour').dropna().reset_index(drop=True)
    features = ['shourlyO3']
    target = collocation_data['rhourlyO3']
    return LinearRegression().fit(collocation_data[features], target)


    # inner join (default) - only keep rows that are in both dataframes
    # merged = pd.merge(temp, h_regular, on='dayhour').dropna()
    # X = create_enhanced_features(merged)
    # feature_cols = ['o3', 'rh', 'temp', 'o3_x_rh', 'o3_x_temp', 
    #                    'rh_x_temp', 'o3_sq', 'rh_sq', 'temp_sq']
    # target = merged['rhourlyO3']

    # scalar = StandardScaler()
    # X_scaled = scalar.fit_transform(X[feature_cols])
    # model = LinearRegression().fit(X_scaled, target)

    # return {
    #     'model': model,
    #     'scaler': scalar,
    #     'feature_cols': feature_cols
    # }

def train_daily_model(sensor_collocation_data, d_regular):
    temp = sensor_collocation_data.groupby('day').agg(
        sdailyO3=('o3', lambda x: x.mean(skipna=True)),
        sdailyRH=('rh', lambda x: x.mean(skipna=True)),
        sdailyTemp=('temp', lambda x: x.mean(skipna=True))
    ).reset_index()

    # inner join (default) - only keep rows that are in both dataframes
    collocation_data = pd.merge(d_regular, temp, on='day').dropna().reset_index(drop=True)
    features = ['sdailyO3']
    target = collocation_data['rdailyO3']
    return LinearRegression().fit(collocation_data[features], target)

    # # inner join (default) - only keep rows that are in both dataframes
    # merged = pd.merge(temp, d_regular, on='day').dropna().sort_values('day')
    # X = create_enhanced_features(merged, hourly=False)
    # feature_cols = ['o3', 'rh', 'temp', 'o3_x_rh', 'o3_x_temp', 
    #                    'rh_x_temp', 'o3_sq', 'rh_sq', 'temp_sq']
    # target = merged['rdailyO3']

    # scalar = StandardScaler()
    # X_scaled = scalar.fit_transform(X[feature_cols])
    # model = LinearRegression().fit(X_scaled, target)

    # return {
    #     'model': model,
    #     'scaler': scalar,
    #     'feature_cols': feature_cols
    # }
    



In [107]:
quantAQ00589 = preprocess_QuantAQ("../ShortenedData/MOD-00589-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00590 = preprocess_QuantAQ("../ShortenedData/MOD-00590-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00591 = preprocess_QuantAQ("../ShortenedData/MOD-00591-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00592 = preprocess_QuantAQ("../ShortenedData/MOD-00592-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00593 = preprocess_QuantAQ("../ShortenedData/MOD-00593-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00595 = preprocess_QuantAQ("../ShortenedData/MOD-00595-RAW.csv", '2025-06-01', '2025-07-14')

In [108]:
# load and clean GAPA data
gapa = pd.read_csv('CO&O3.csv', header=2)
gapa.columns = ['Date', 'O3', 'CO']
gapa = gapa[['Date', 'O3']]

gapa['time'] = pd.to_datetime(gapa['Date'], format='%m/%d/%Y %H:%M')
gapa['day'] = gapa['time'].dt.date
gapa['dayhour'] = gapa['time'].dt.strftime('%Y-%m-%d %H')
gapa.rename(columns={'O3':'ro3'}, inplace=True)


gapa['ro3'] = gapa['ro3'] * 1000

# Sort chronologically
gapa = gapa.sort_values('time').reset_index(drop=True).dropna(subset=['ro3'])

h_gapa = gapa.groupby('dayhour').agg(
    rhourlyO3=('ro3', lambda x: x.mean(skipna=True)),
).reset_index()

d_gapa = gapa.groupby('day').agg(
    rdailyO3=('ro3', lambda x: pd.to_numeric(x, errors='coerce').mean(skipna=True)),
).reset_index()




In [109]:
hmodel_00589 = train_hourly_model(quantAQ00589, h_gapa)
dmodel_00589 = train_daily_model(quantAQ00589, d_gapa)
hmodel_00590 = train_hourly_model(quantAQ00590, h_gapa)
dmodel_00590 = train_daily_model(quantAQ00590, d_gapa)
hmodel_00591 = train_hourly_model(quantAQ00591, h_gapa)
dmodel_00591 = train_daily_model(quantAQ00591, d_gapa)
hmodel_00592 = train_hourly_model(quantAQ00592, h_gapa)
dmodel_00592 = train_daily_model(quantAQ00592, d_gapa)
hmodel_00593 = train_hourly_model(quantAQ00593, h_gapa)
dmodel_00593 = train_daily_model(quantAQ00593, d_gapa)
hmodel_00595 = train_hourly_model(quantAQ00595, h_gapa)
dmodel_00595 = train_daily_model(quantAQ00595, d_gapa)

In [110]:
# save models
joblib.dump(hmodel_00589, 'models/hmodel_00589.joblib')
joblib.dump(dmodel_00589, 'models/dmodel_00589.joblib')
joblib.dump(hmodel_00590, 'models/hmodel_00590.joblib')
joblib.dump(dmodel_00590, 'models/dmodel_00590.joblib')
joblib.dump(hmodel_00591, 'models/hmodel_00591.joblib')
joblib.dump(dmodel_00591, 'models/dmodel_00591.joblib')
joblib.dump(hmodel_00592, 'models/hmodel_00592.joblib')
joblib.dump(dmodel_00592, 'models/dmodel_00592.joblib')
joblib.dump(hmodel_00593, 'models/hmodel_00593.joblib')
joblib.dump(dmodel_00593, 'models/dmodel_00593.joblib')
joblib.dump(hmodel_00595, 'models/hmodel_00595.joblib')
joblib.dump(dmodel_00595, 'models/dmodel_00595.joblib')


['models/dmodel_00595.joblib']